# SIC8 | 04 — Historical Fraud Detection

**Goal:** Detect suspicious behavior in validated historical Silver transactions using explainable Spark rules, then evaluate predictions against isolated historical labels. **No streaming, no Gold publication, and no Raw/Silver changes by default.**

**Scope:** Sliding 10-minute **card-level** CNP bursts and micro-testing. Historical `account_id` and exact GPS are unavailable, so account velocity, geographical velocity, and terminal-failure KPIs are **not claimed as verified** here.

Run `FIXTURE` to verify rule behavior; run `SILVER` only after Notebook 03 has published and validated its tables. All thresholds are **initial project hypotheses**, not optimized fraud-policy settings.

## 01 — Settings
Choose a deterministic rule-test fixture or the complete published Silver dataset. Writes are disabled until both ID approval and result review.

In [0]:
from pyspark.sql import functions as F, Window
from decimal import Decimal
import time

RUN_MODE = "FIXTURE"           # "FIXTURE" or "SILVER" (entire historical snapshot)
HISTORICAL_IDS_APPROVED = False # True only after Notebook 03 ID/key review
ENABLE_WRITE = False           # No managed-table writes by default

SILVER_SCHEMA = "workspace.sic8_silver"
TX_TABLE = f"{SILVER_SCHEMA}.historical_transactions"
LABEL_TABLE = f"{SILVER_SCHEMA}.historical_ground_truth"

RULESET_VERSION = "hist-rules-0.1"
WINDOW_MINUTES = 10
WINDOW_SECONDS = WINDOW_MINUTES * 60
CNP_BURST_MIN_COUNT = 5
MICRO_MAX_AMOUNT = Decimal("5.00")
MICRO_MIN_COUNT = 4

if RUN_MODE not in ("FIXTURE", "SILVER"):
    raise ValueError("RUN_MODE must be FIXTURE or SILVER")
if RUN_MODE == "SILVER" and not HISTORICAL_IDS_APPROVED:
    raise RuntimeError("Review Notebook 03 source IDs and publish stable Silver tables first")
if ENABLE_WRITE and RUN_MODE != "SILVER":
    raise ValueError("Do not publish test fixtures as real detection output")
print("Run mode:", RUN_MODE, "| Version:", RULESET_VERSION, "| Write:", ENABLE_WRITE)

Run mode: FIXTURE | Version: hist-rules-0.1 | Write: False


## 02 — Rule-test fixture
A tiny **synthetic unit-test fixture**, not a sample or result from Kaggle. Expected: 2 burst alerts, 1 micro-testing alert, and no POS alerts.

In [0]:
fixture_rows = [
    # Six online transactions for the same card in six minutes: last two trigger burst.
    (f"BURST_{i}", "TEST_CARD_1", "TEST_USER_1", f"2020-01-01 12:0{i}:00", "20.00", "ONLINE", "ECOMMERCE", False, "USD")
    for i in range(6)
] + [
    # Four small online payments for another card: last one triggers micro-testing.
    (f"MICRO_{i}", "TEST_CARD_2", "TEST_USER_2", f"2020-01-01 13:0{i}:00", f"{i + 1}.00", "ONLINE", "ECOMMERCE", False, "USD")
    for i in range(4)
] + [
    ("POS_1", "TEST_CARD_3", "TEST_USER_3", "2020-01-01 14:00:00", "50.00", "SWIPE", "POS", True, "USD"),
    ("POS_2", "TEST_CARD_3", "TEST_USER_3", "2020-01-01 14:01:00", "50.00", "SWIPE", "POS", True, "USD"),
]

fixture_base = spark.createDataFrame(
    fixture_rows,
    "transaction_id string, card_id string, customer_id string, local_time_text string, "
    "amount_text string, payment_method string, channel string, card_present boolean, currency string"
)
fixture_transactions = (fixture_base
    .withColumn("source_event_time_local", F.expr("try_cast(local_time_text AS TIMESTAMP_NTZ)"))
    .withColumn("amount", F.expr("try_cast(amount_text AS DECIMAL(18,2))"))
    .withColumn("source_system", F.lit("EXPLICIT_TEST_FIXTURE"))
    .withColumn("event_id", F.concat(F.lit("FIX_EVENT_"), F.col("transaction_id")))
    .withColumn("channel_origin", F.lit("FIXTURE_DEFINED"))
    .withColumn("event_time_timezone_status", F.lit("UNKNOWN"))
    .withColumn("geo_eligible_for_exact_velocity", F.lit(False))
    .drop("local_time_text", "amount_text"))

fixture_labels = spark.createDataFrame(
    [(row[0], row[0] in {"BURST_4", "BURST_5", "MICRO_3"}) for row in fixture_rows],
    "transaction_id string, ground_truth_label boolean"
)
print("Fixture: 12 synthetic rows; expected 3 flagged transactions / 3 alerts")

Fixture: 12 synthetic rows; expected 3 flagged transactions / 3 alerts


## 03 — Load published Silver or fixture
The detector receives **transactions only**. Ground-truth labels are loaded separately and never joined until evaluation.

In [0]:
if RUN_MODE == "FIXTURE":
    transactions = fixture_transactions
    truth = fixture_labels
else:
    for table in (TX_TABLE, LABEL_TABLE):
        if not spark.catalog.tableExists(table):
            raise RuntimeError(f"Missing {table}. Publish and validate Notebook 03 first.")
    transactions = spark.table(TX_TABLE)
    truth = spark.table(LABEL_TABLE).select("transaction_id", "ground_truth_label")

required_tx = {
    "transaction_id", "event_id", "card_id", "customer_id", "source_event_time_local",
    "amount", "currency", "payment_method", "card_present", "channel",
    "event_time_timezone_status", "geo_eligible_for_exact_velocity", "source_system"
}
missing = required_tx - set(transactions.columns)
if missing:
    raise ValueError(f"Missing Silver transaction columns: {sorted(missing)}")
if not {"transaction_id", "ground_truth_label"}.issubset(truth.columns):
    raise ValueError("Ground-truth table does not meet the expected schema")
if "ground_truth_label" in transactions.columns or "Is Fraud?" in transactions.columns:
    raise AssertionError("Ground-truth leakage into detection features")
print("Detector contract: OK | Source:", "fixture" if RUN_MODE == "FIXTURE" else TX_TABLE)

Detector contract: OK | Source: fixture


## 04 — Input gates
Rule scope is **historical USD**. Unknown timestamps, card IDs, and money types stop the run instead of silently producing bad alerts.

In [0]:
if dict(transactions.dtypes).get("source_event_time_local") != "timestamp_ntz":
    raise TypeError("Expected source_event_time_local TIMESTAMP_NTZ; review Silver contract")
if not dict(transactions.dtypes).get("amount", "").startswith("decimal("):
    raise TypeError("Expected DECIMAL amount; review Silver contract")

input_quality = transactions.agg(
    F.count("*").alias("rows"),
    F.sum(F.col("transaction_id").isNull().cast("long")).alias("null_ids"),
    F.sum(F.col("card_id").isNull().cast("long")).alias("null_cards"),
    F.sum(F.col("source_event_time_local").isNull().cast("long")).alias("null_times"),
    F.sum(F.col("amount").isNull().cast("long")).alias("null_amounts"),
    F.sum((F.col("currency") != "USD").cast("long")).alias("other_currency")
).first().asDict()
print("Source quality:", input_quality)
if any((input_quality[k] or 0) != 0 for k in
       ("null_ids", "null_cards", "null_times", "null_amounts", "other_currency")):
    raise ValueError("Input quality gate failed. Investigate Silver before detection.")

# The original snapshot ID review is required separately; do not mistake row counts for uniqueness.
if RUN_MODE == "FIXTURE":
    assert transactions.select("transaction_id").distinct().count() == input_quality["rows"]
print("Quality gates: PASS")

Source quality: {'rows': 12, 'null_ids': 0, 'null_cards': 0, 'null_times': 0, 'null_amounts': 0, 'other_currency': 0}
Quality gates: PASS


## 05 — Build a local-time numeric key
Create **naive local wall-clock seconds**, not UTC epoch seconds. This supports a historical within-card approximation without assigning an invented timezone.

In [0]:
event_ts = F.col("source_event_time_local")
local_seconds = (
    F.datediff(F.to_date(event_ts), F.lit("1970-01-01").cast("date")).cast("long") * F.lit(86400)
    + F.hour(event_ts) * F.lit(3600)
    + F.minute(event_ts) * F.lit(60)
    + F.second(event_ts)
)
features = (transactions
    .withColumn("local_event_seconds", local_seconds)
    .withColumn("is_cnp", (F.col("payment_method") == "ONLINE") & (F.col("card_present") == False))
    .withColumn("is_micro_payment", F.col("is_cnp") &
                (F.col("amount") > F.lit(Decimal("0.00"))) &
                (F.col("amount") <= F.lit(MICRO_MAX_AMOUNT))))

print("Derived CNP/micro indicators; no source or timestamps were modified.")

Derived CNP/micro indicators; no source or timestamps were modified.


## 06 — Rolling 10-minute windows
Use a **time range** rather than `rowsBetween`: count matching transactions within [t - 600 s, t] per card. Same-timestamp peers share a window.

In [0]:
rolling_10m = (
    Window.partitionBy("card_id", "currency")
    .orderBy(F.col("local_event_seconds"))
    .rangeBetween(-WINDOW_SECONDS, 0)
)

scored = (features
    .withColumn("cnp_count_10m", F.sum(F.col("is_cnp").cast("long")).over(rolling_10m))
    .withColumn("micro_count_10m", F.sum(F.col("is_micro_payment").cast("long")).over(rolling_10m))
    .withColumn("flag_cnp_burst", F.col("is_cnp") &
                (F.col("cnp_count_10m") >= F.lit(CNP_BURST_MIN_COUNT)))
    .withColumn("flag_micro_testing", F.col("is_micro_payment") &
                (F.col("micro_count_10m") >= F.lit(MICRO_MIN_COUNT)))
    .withColumn("is_flagged", F.col("flag_cnp_burst") | F.col("flag_micro_testing"))
    .withColumn("geo_rule_status", F.lit("NOT_EVALUATED_NO_VERIFIED_GPS_UTC"))
    .withColumn("ruleset_version", F.lit(RULESET_VERSION)))

print("Rolling card-level rules defined. Full Spark window executes on an action.")

Rolling card-level rules defined. Full Spark window executes on an action.


## 07 — Generate explainable alerts
One row per `(transaction_id, rule_id)`. Two rules on one transaction produce two alerts, but downstream transaction counts must remain distinct.

In [0]:
ALERT_FIELDS = [
    "transaction_id", "event_id", "card_id", "customer_id", "source_event_time_local",
    "currency", "amount", "source_system", "ruleset_version"
]

burst_alerts = (scored.filter(F.col("flag_cnp_burst"))
    .select(*ALERT_FIELDS, F.lit("RAPID_CNP_BURST").alias("rule_id"),
            F.col("cnp_count_10m").alias("observed_count"),
            F.lit(CNP_BURST_MIN_COUNT).alias("threshold_count")))

micro_alerts = (scored.filter(F.col("flag_micro_testing"))
    .select(*ALERT_FIELDS, F.lit("MICRO_TESTING").alias("rule_id"),
            F.col("micro_count_10m").alias("observed_count"),
            F.lit(MICRO_MIN_COUNT).alias("threshold_count")))

fraud_alerts = (burst_alerts.unionByName(micro_alerts)
    .withColumn("alert_id", F.sha2(F.concat_ws("|", "transaction_id", "rule_id", "ruleset_version"), 256))
    .withColumn("window_minutes", F.lit(WINDOW_MINUTES))
    .withColumn("evaluation_scope", F.lit("HISTORICAL_CARD_LEVEL_LOCAL_TIME")))

score_output = scored.select(
    "transaction_id", "event_id", "card_id", "customer_id", "source_event_time_local",
    "currency", "amount", "channel", "payment_method", "source_system", "cnp_count_10m",
    "micro_count_10m", "flag_cnp_burst", "flag_micro_testing", "is_flagged",
    "geo_rule_status", "ruleset_version"
)
print("Scored transactions and fraud alerts prepared in memory.")

Scored transactions and fraud alerts prepared in memory.


## 08 — Evaluate AFTER scoring
Labels are attached only to the scored output. Recall (`TP / (TP + FN)`) measures detection of the historical source labels, **not live fraud detection rate**.

In [0]:
# Verify one label per transaction before joining; this matters for correct KPI denominators.
if truth.groupBy("transaction_id").count().filter(F.col("count") != 1).limit(1).count():
    raise ValueError("Duplicate ground-truth transaction IDs")
if truth.filter(F.col("transaction_id").isNull()).limit(1).count():
    raise ValueError("Null transaction ID in ground truth")

predictions_with_truth = score_output.join(truth, "transaction_id", "left")
confusion = predictions_with_truth.agg(
    F.count("*").alias("evaluated_rows"),
    F.sum(F.col("ground_truth_label").isNull().cast("long")).alias("missing_labels"),
    F.sum((F.col("is_flagged") & F.col("ground_truth_label")).cast("long")).alias("tp"),
    F.sum((F.col("is_flagged") & ~F.col("ground_truth_label")).cast("long")).alias("fp"),
    F.sum((~F.col("is_flagged") & F.col("ground_truth_label")).cast("long")).alias("fn"),
    F.sum((~F.col("is_flagged") & ~F.col("ground_truth_label")).cast("long")).alias("tn"),
    F.sum(F.col("flag_cnp_burst").cast("long")).alias("burst_alert_count"),
    F.sum(F.col("flag_micro_testing").cast("long")).alias("micro_alert_count"),
    F.sum(F.col("is_flagged").cast("long")).alias("flagged_transaction_count")
).first().asDict()

if confusion["missing_labels"] != 0:
    raise ValueError("Missing ground-truth labels after transaction ID join")
if sum(confusion[k] or 0 for k in ("tp", "fp", "fn", "tn")) != confusion["evaluated_rows"]:
    raise AssertionError("Confusion-matrix reconciliation failed")

tp, fp, fn, tn = (confusion[k] or 0 for k in ("tp", "fp", "fn", "tn"))
precision = tp / (tp + fp) if tp + fp else None
recall = tp / (tp + fn) if tp + fn else None
print("Historical evaluation:", confusion)
print("Precision:", precision, "| Recall:", recall)
print("WARNING: These are uncalibrated historical heuristic rules, not live detection metrics.")

Historical evaluation: {'evaluated_rows': 12, 'missing_labels': 0, 'tp': 3, 'fp': 0, 'fn': 0, 'tn': 9, 'burst_alert_count': 2, 'micro_alert_count': 1, 'flagged_transaction_count': 3}
Precision: 1.0 | Recall: 1.0


## 09 — Fixture assertions and real-run summary
In fixture mode, hard assertions verify expected alerts and label evaluation. In Silver mode, report actual counts (which may be expensive to compute).

In [0]:
if RUN_MODE == "FIXTURE":
    alert_counts = {r["rule_id"]: r["count"] for r in fraud_alerts.groupBy("rule_id").count().collect()}
    flagged_ids = {r["transaction_id"] for r in score_output.filter(F.col("is_flagged"))
                   .select("transaction_id").collect()}
    assert alert_counts == {"RAPID_CNP_BURST": 2, "MICRO_TESTING": 1}, alert_counts
    assert flagged_ids == {"BURST_4", "BURST_5", "MICRO_3"}, flagged_ids
    assert (tp, fp, fn, tn) == (3, 0, 0, 9), (tp, fp, fn, tn)
    print("FIXTURE TESTS: PASS | 12 rows, 3 alerts, 3 flagged transactions")
else:
    total_alerts = (confusion["burst_alert_count"] or 0) + (confusion["micro_alert_count"] or 0)
    distinct_alerted_transactions = confusion["flagged_transaction_count"] or 0
    print("Historical transactions evaluated:", confusion["evaluated_rows"])
    print("Alert rows:", total_alerts, "| Distinct flagged transactions:", distinct_alerted_transactions)
    print("Geo-velocity and terminal-failure KPIs: NOT EVALUATED for historical source")

FIXTURE TESTS: PASS | 12 rows, 3 alerts, 3 flagged transactions


## 10 — Safe preview
Preview a few alerts without card PAN, CVV, or isolated labels. Preview is not a full-dataset quality test.

In [0]:
display(fraud_alerts.select(
    "alert_id", "transaction_id", "rule_id", "source_event_time_local",
    "observed_count", "threshold_count", "ruleset_version"
).limit(10))
print("Alert preview complete; no tables written.")

alert_id,transaction_id,rule_id,source_event_time_local,observed_count,threshold_count,ruleset_version
b71d612d85a7907856b006daeb641d81110e99549904faa276801deb4f4b6412,BURST_4,RAPID_CNP_BURST,2020-01-01T12:04:00.000,5,5,hist-rules-0.1
47fd5fd9d7c4ba2a01fb0cc3737c8cb1408d16f84eaa135d44587d3a5e5b12f4,BURST_5,RAPID_CNP_BURST,2020-01-01T12:05:00.000,6,5,hist-rules-0.1
c7a62ffdbe308fd8f071db185b143b488e2f6d3f7abba7366811526a4a76c5d0,MICRO_3,MICRO_TESTING,2020-01-01T13:03:00.000,4,4,hist-rules-0.1


Alert preview complete; no tables written.


## 11 — Optional Delta publication
Disabled by default. Publishing overwrites **historical-only output tables** after review; it does not modify Silver source tables. Writes to different tables are not a single atomic transaction.

In [0]:
if ENABLE_WRITE:
    if RUN_MODE != "SILVER" or not HISTORICAL_IDS_APPROVED:
        raise RuntimeError("Production publishing requires reviewed historical Silver IDs")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")
    outputs = {
        f"{SILVER_SCHEMA}.historical_fraud_scores": score_output,
        f"{SILVER_SCHEMA}.historical_fraud_alerts": fraud_alerts,
    }
    for name, frame in outputs.items():
        (frame.write.format("delta").mode("overwrite")
            .option("overwriteSchema", "true").saveAsTable(name))
        print("Published:", name)
else:
    print("DRY RUN: no Delta tables or S3 objects changed.")

DRY RUN: no Delta tables or S3 objects changed.


## 12 — Handoff and limitations
**Outputs:** `score_output`, `fraud_alerts`, historical evaluation metrics. Next: Gold/SQL modeling after reviewed Silver and verified rule results; later reuse the rules on the streaming canonical contract.

**Known limitations:** (1) within-card local-time approximation is not global/UTC event time; (2) historical `account_id`, GPS and reliable terminal outcomes are absent; (3) Chip → ATM is an explicit project assumption and is not used as a fraud signal; (4) the thresholds need tuning and false-positive review; (5) stable snapshot IDs must be approved separately.

In [0]:
print("Run mode:", RUN_MODE)
print("Source:", "local test fixture" if RUN_MODE == "FIXTURE" else TX_TABLE)
print("Rule scope: CARD-level | local-time 10-minute rolling window")
print("Outputs: score_output, fraud_alerts, historical confusion metrics")
print("Geo velocity: NOT_EVALUATED | Terminal failures: NOT_EVALUATED")
print("Published:", bool(ENABLE_WRITE))

Run mode: FIXTURE
Source: local test fixture
Rule scope: CARD-level | local-time 10-minute rolling window
Outputs: score_output, fraud_alerts, historical confusion metrics
Geo velocity: NOT_EVALUATED | Terminal failures: NOT_EVALUATED
Published: False
